# LLaVA-Interleave (0.5B) — ZERO-SHOT EXPERIMENT A ONLY (with explicit density definition)

This notebook runs Experiment A only (no YOLO count) — for Experiment B, use `LLaVA_ExperimentB_ZeroShot_DensityDef.ipynb`.

Same model, same technical setup (`min_new_tokens=18`, digit-length-bounded parser) as the original LLaVA zero-shot Experiment A notebook. The only change: the system prompt's density-labels section now defines what Low/Medium/High mean visually. Outputs save with a `_densitydef` suffix so the original results aren't overwritten.

**Note going in:** the original zero-shot A run showed severe format-reliability collapse (61.4% failure) and near-total reliance on round-number placeholders even in correctly-formatted responses — a more fundamental grounding problem than category ambiguity. This definition may not move the needle much here; worth checking rather than assuming.

## 1. Mount Drive & confirm input data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/CrowdDensityResearch/Phase5_Input.zip"

if not os.path.exists("/content/Phase5_Input"):
    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("Input ready:", os.path.exists("/content/Phase5_Input"))

## 2. Back up any existing zero-shot results before overwriting

In [ ]:
import os, shutil

RESULT_DIR = "/content/drive/MyDrive/CrowdDensityResearch/VLM_Results"
os.makedirs(RESULT_DIR, exist_ok=True)

for fname in ["llava_experiment_A_zeroshot_densitydef.csv", "llava_experiment_A_zeroshot_densitydef_evaluated.csv"]:
    src = os.path.join(RESULT_DIR, fname)
    if os.path.exists(src):
        backup = os.path.join(RESULT_DIR, fname.replace(".csv", "_backup.csv"))
        if not os.path.exists(backup):
            shutil.copy2(src, backup)
            print(f"Backed up {fname} -> {os.path.basename(backup)}")
        else:
            print(f"Backup already exists for {fname}, skipping.")
    else:
        print(f"No existing {fname} found — nothing to back up.")

## 3. Install packages

In [ ]:
!pip install -q -U transformers accelerate

## 4. Load the model (identical to the original LLaVA experiments)

In [ ]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL_NAME = "llava-hf/llava-interleave-qwen-0.5b-hf"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

print("Loading model...")
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print("✅ Model loaded successfully!")

## 5. Helpers

In [ ]:
from PIL import Image

def load_image(path, max_size=512):
    """Load an image and resize it to reduce memory usage."""
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_size, max_size))
    return img

DENSITY_TEXT = {
    0: "Low",
    1: "Medium",
    2: "High"
}

## 6. Zero-shot prompt builder — Experiment A
Same system prompt text used for Qwen's zero-shot experiments, adapted to LLaVA's message format (image referenced by placeholder, actual PIL image passed separately).

In [ ]:
def build_messages_zeroshot_A(test_image):
    """LLaVA conversation for one test image — zero-shot Experiment A. No support images.
    Returns (conversation, images) — images is a list matching each {"type": "image"}
    placeholder in the conversation, same convention as the original LLaVA notebooks."""
    conversation = []
    images = []

    conversation.append({
        "role": "system",
        "content": [{
            "type": "text",
            "text": """You are an expert in crowd counting and crowd density estimation.

You will receive one image.

Predict:

1. The EXACT crowd count.
2. The crowd density.

Density labels — judge by how visually packed the crowd appears, not by exact count:

0 = Low: People are clearly separated, with visible space between individuals. Each person is easy to distinguish on their own.

1 = Medium: People are moderately close together. Some overlap between individuals, but gaps and open space are still visible.

2 = High: People are tightly packed with little or no visible space between them. Individuals often overlap and may be hard to distinguish separately.

Return ONLY:

Count: <integer>
Density: <0/1/2>

Do not explain.
Do not estimate.
Do not output ranges.
Return exactly one integer count.
"""
        }]
    })

    conversation.append({
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": """TEST IMAGE

Count every visible person exactly once.

Return ONLY

Count: <integer>

Density: <0/1/2>
"""}
        ]
    })
    images.append(load_image(test_image))

    return conversation, images

## 7. Inference and parsing

In [ ]:
import gc
import re
import torch


def run_llava(conversation, images):
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)

    inputs = processor(images=images, text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device, torch.float16) if v.dtype.is_floating_point else v.to(model.device)
              for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs, max_new_tokens=40, min_new_tokens=18, do_sample=False
        )

    trimmed_ids = generated_ids[:, inputs["input_ids"].shape[1]:]
    response = processor.batch_decode(
        trimmed_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    del inputs, generated_ids, trimmed_ids
    gc.collect()
    torch.cuda.empty_cache()

    return response.strip()


def parse_response(response):
    """Rejects degenerate long digit-runs as unparseable rather than
    taking them literally (same guard used for the final LLaVA runs)."""
    response = str(response)

    count = None
    m = re.search(r"Count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if m:
        digits = m.group(1)
        if len(digits) <= 5:
            count = int(digits)

    density = None
    m = re.search(r"Density\s*[:=]\s*([012])", response, re.IGNORECASE)
    if m:
        density = int(m.group(1))
    else:
        r = response.lower()
        if "low" in r:
            density = 0
        elif "medium" in r:
            density = 1
        elif "high" in r:
            density = 2

    return count, density

## 8. Load test data
No `support_mapping.csv` needed at all for zero-shot.

In [ ]:
import pandas as pd

ROOT = "/content/Phase5_Input"
TEST_DIR = f"{ROOT}/test/images"

test_df = pd.read_csv(f"{ROOT}/test.csv")
print("Test images:", len(test_df))

## 9. Smoke test — check 3 samples before running the full loop

In [ ]:
N_SMOKE = 3

for idx in range(N_SMOKE):
    sample_i = test_df.iloc[idx]
    test_image_i = os.path.join(TEST_DIR, f"{int(sample_i['filename']):04d}.jpg")

    conversation_i, images_i = build_messages_zeroshot_A(test_image_i)
    response_i = run_llava(conversation_i, images_i)
    parsed_i = parse_response(response_i)

    print(f"--- Sample {idx} (filename={int(sample_i['filename'])}, true_count={int(sample_i['total_count'])}) ---")
    print("Response:", repr(response_i))
    print("Parsed  :", parsed_i)
    if parsed_i[0] is None or parsed_i[1] is None:
        print("⚠️  Incomplete parse.")
    print()

## 10. Full run — all 1600 test images, saved continuously to Drive

In [ ]:
from tqdm import tqdm
import os
import gc
import time
import torch

OUT_PATH = os.path.join(RESULT_DIR, "llava_experiment_A_zeroshot_densitydef.csv")

if "results_A_zs" not in globals():
    if os.path.exists(OUT_PATH):
        prior = pd.read_csv(OUT_PATH)
        results_A_zs = prior.to_dict("records")
        for r in results_A_zs:
            r["test_filename"] = int(r["test_filename"])
        print(f"Resumed {len(results_A_zs)} results from a previous session's CSV: {OUT_PATH}")
    else:
        results_A_zs = []

done_filenames = {r["test_filename"] for r in results_A_zs}
remaining = test_df[~test_df["filename"].astype(int).isin(done_filenames)]
print(f"Already done: {len(done_filenames)} | Remaining: {len(remaining)}")

MAX_RETRIES_PER_IMAGE = 2

for _, sample in tqdm(remaining.iterrows(), total=len(remaining)):

    test_image = os.path.join(TEST_DIR, f"{int(sample['filename']):04d}.jpg")

    conversation, images = build_messages_zeroshot_A(test_image)

    response = None
    for attempt in range(1, MAX_RETRIES_PER_IMAGE + 1):
        try:
            response = run_llava(conversation, images)
            break
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            print(f"\n[OOM] filename={int(sample['filename'])} "
                  f"attempt {attempt}/{MAX_RETRIES_PER_IMAGE} — clearing cache and retrying...")
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(2)

    if response is None:
        pred_count, pred_density = None, None
        response = "ERROR: CUDA out of memory after retries"
    else:
        pred_count, pred_density = parse_response(response)

    results_A_zs.append({
        "test_filename": int(sample["filename"]),
        "predicted_count": pred_count,
        "predicted_density": pred_density,
        "raw_response": response
    })

    pd.DataFrame(results_A_zs).to_csv(OUT_PATH, index=False)

failed = sum(1 for r in results_A_zs if r["predicted_count"] is None)
print("Finished:", len(results_A_zs), f"({failed} unparseable)")
print("Saved to:", OUT_PATH)

## 11. Evaluate — counting metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

llava_A_zs = pd.DataFrame(results_A_zs)

llava_A_zs_eval = llava_A_zs.merge(
    test_df[["filename", "total_count", "crowd_density"]],
    left_on="test_filename",
    right_on="filename",
    how="left"
)

print("Missing ground truth after merge:", llava_A_zs_eval["total_count"].isna().sum())

n_before = len(llava_A_zs_eval)
count_valid = llava_A_zs_eval.dropna(subset=["predicted_count"]).copy()
n_dropped = n_before - len(count_valid)
print(f"Dropped {n_dropped}/{n_before} rows with no parseable count ({100*n_dropped/n_before:.1f}%)")

y_true_count = count_valid["total_count"]
y_pred_count = count_valid["predicted_count"]

mae = mean_absolute_error(y_true_count, y_pred_count)
rmse = np.sqrt(mean_squared_error(y_true_count, y_pred_count))

nonzero = count_valid["total_count"] > 0
mape = np.mean(np.abs(
    (count_valid.loc[nonzero, "total_count"] - count_valid.loc[nonzero, "predicted_count"])
    / count_valid.loc[nonzero, "total_count"]
)) * 100

print("LLaVA Experiment A (Zero-Shot) — Counting")
print("===========================================")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"MAPE : {mape:.2f}%")
print(f"(computed on {len(count_valid)}/{n_before} images with a parseable count)")

## 12. Evaluate — density classification

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

density_valid = llava_A_zs_eval.dropna(subset=["predicted_density"]).copy()
n_density_dropped = len(llava_A_zs_eval) - len(density_valid)
print(f"Dropped {n_density_dropped}/{len(llava_A_zs_eval)} rows with no parseable density ({100*n_density_dropped/len(llava_A_zs_eval):.1f}%)")

y_true_density = density_valid["crowd_density"]
y_pred_density = density_valid["predicted_density"]

accuracy = accuracy_score(y_true_density, y_pred_density)
precision = precision_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
recall = recall_score(y_true_density, y_pred_density, average="weighted", zero_division=0)
f1 = f1_score(y_true_density, y_pred_density, average="weighted", zero_division=0)

print("LLaVA Experiment A (Zero-Shot) — Density")
print("==========================================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(y_true_density, y_pred_density, labels=[0, 1, 2])
print()
print("Confusion matrix (rows=GT, cols=pred):")
print(cm)

## 13. Save evaluated results to Drive

In [ ]:
EVAL_PATH = os.path.join(RESULT_DIR, "llava_experiment_A_zeroshot_densitydef_evaluated.csv")
llava_A_zs_eval.to_csv(EVAL_PATH, index=False)

print("Saved:", len(llava_A_zs_eval), "rows")
print("Path:", EVAL_PATH)